# Static Move Prediction Draft

## Setup Environment && Load Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dataclasses import dataclass

import mlflow
import optuna

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight

from chesswinnerprediction.static_move.models.static_move_base import StaticMoveBaseModel

from config import RANDOM_STATE
from chesswinnerprediction.static_move.utils import load_train_valid_test, setup_mlflow, log_prediction

## Setup MLflow

In [3]:
setup_mlflow(experiment_name="Static Move Prediction")

## Load Data

In [4]:
X_train, y_train, X_valid, y_valid, X_test, y_test = load_train_valid_test(drop_event=True)

train_sample_weight = X_train["sample_weight"]
X_train.drop(columns=["sample_weight"], inplace=True)

valid_sample_weight = X_valid["sample_weight"]
X_valid.drop(columns=["sample_weight"], inplace=True)

In [5]:
eval_and_is_checkmate = {
    X_train.columns.get_loc("i_move"),
    X_train.columns.get_loc("eval"),
    X_train.columns.get_loc("is_checkmate_countdown"),
}
board_features = {
    X_train.columns.get_loc("i_move"),
    X_train.columns.get_loc("w_score"),
    X_train.columns.get_loc("b_score"),
    X_train.columns.get_loc("n_pieces"),
}
eval_and_board = eval_and_is_checkmate.union(board_features)

time_control = {
    X_train.columns.get_loc("i_move"),
    X_train.columns.get_loc("GameDurations_norm"),
    X_train.columns.get_loc("black_remaining_time_norm"),
    X_train.columns.get_loc("white_remaining_time_norm"),
    X_train.columns.get_loc("time_diff_norm"),
    X_train.columns.get_loc("IncrementTime"),
    X_train.columns.get_loc("BaseTime"),
}

time_control_extra = {
    X_train.columns.get_loc("black_time_will_end_on_move"),
    X_train.columns.get_loc("white_time_will_end_on_move"),
    X_train.columns.get_loc("black_increment_pct_in_time_per_move"),
    X_train.columns.get_loc("black_increment_pct_in_time_per_move"),
    X_train.columns.get_loc("black_time_per_move"),
    X_train.columns.get_loc("white_time_per_move"),
    X_train.columns.get_loc("i_move"),
}

all_features = set(range(X_train.shape[1]))
other_features = all_features.difference(eval_and_board, time_control, time_control_extra)

interaction_cst = [
    eval_and_is_checkmate,
    board_features,
    eval_and_board,
    time_control,
    time_control_extra,
    other_features
]

In [6]:
X_test.head(10)

,eval,EloDiff,MeanElo,BaseTime,IncrementTime,time_diff_norm,white_remaining_time_norm,black_remaining_time_norm,GameDurations_norm,is_checkmate_countdown,i_move,w_score,b_score,n_pieces,white_time_per_move,black_time_per_move,white_increment_pct_in_time_per_move,black_increment_pct_in_time_per_move,white_time_will_end_on_move,black_time_will_end_on_move
0,1.77,-191,1445.5,600,0,-0.025000,0.726667,0.751667,0.521667,False,45,28,28,22,0.006075,0.005520,0.000000,0.000000,119.614454,136.183376
1,-6.00,282,1304.0,300,1,0.076667,0.633333,0.556667,0.993333,True,57,14,24,16,0.006434,0.007779,0.518101,0.428516,98.439243,71.562228
2,-0.10,-146,1980.0,60,0,0.066667,0.966667,0.900000,0.133333,False,20,38,38,30,0.001668,0.005001,0.000000,0.000000,200.000000,179.964007
3,-6.69,43,1235.5,600,0,-0.045000,0.700000,0.745000,0.555000,False,43,11,15,18,0.006978,0.005931,0.000000,0.000000,100.318954,125.606270
4,-7.14,-401,1894.5,300,0,-0.050000,0.336667,0.386667,1.276667,False,64,10,13,15,0.010366,0.009584,0.000000,0.000000,32.479278,40.343616
5,3.00,-16,1242.0,60,0,-0.083333,0.516667,0.600000,0.883333,True,46,38,1,17,0.010508,0.008697,0.000000,0.000000,49.167734,68.992066
6,5.81,-208,1824.0,600,0,0.066667,0.495000,0.428333,1.076667,False,57,21,19,18,0.008861,0.010030,0.000000,0.000000,55.864982,42.704197
7,-7.65,25,1582.5,60,0,0.050000,0.966667,0.916667,0.116667,False,13,30,36,30,0.002565,0.006411,0.000000,0.000000,200.000000,142.977695
8,-3.53,-33,1134.5,30,0,0.133333,0.166667,0.033333,1.800000,False,42,30,33,27,0.019842,0.023017,0.000000,0.000000,8.399577,1.448213
9,5.76,24,1824.0,180,0,0.027778,0.200000,0.172222,1.627778,False,75,15,12,19,0.010668,0.011038,0.000000,0.000000,18.748242,15.602613


## Optuna

In [7]:
def get_model(trial: optuna.Trial, model_type, hist_gb_class_weight=None):
    if model_type == HistGradientBoostingClassifier.__name__:
        model = HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            tol=1e-5,
            n_iter_no_change=10,
            learning_rate=trial.suggest_float("hist_learning_rate", 5e-4, 1e-2, log=True),
            max_iter=trial.suggest_int("hist_max_iter", 800, 1600),
            max_depth=trial.suggest_int("hist_max_depth", 5, 20),
            min_samples_leaf=trial.suggest_int("hist_min_samples_leaf", 10, X_train.shape[0]//8),
            max_leaf_nodes=trial.suggest_int("hist_max_leaf_nodes", 50, 300),
            # l2_regularization=trial.suggest_float("hist_l2_regularization", 1e-7, 1, log=True),
            max_bins=trial.suggest_int("hist_max_bins", 100, 255),
            # categorical_features=["Event"],
            interaction_cst=interaction_cst,
            max_features=trial.suggest_float("hist_max_features", 0.5, 1.0),
            class_weight=hist_gb_class_weight,
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return model

In [8]:
@dataclass
class BestModel:
    trial: optuna.Trial = None
    score: float = 0.0
    run_id: str = ""
    model: StaticMoveBaseModel = None

In [9]:
class Objective:
    def __init__(self, parent_run_id, model_types, fit_kwargs, get_model_kwargs, score_model_kwargs):
        self.parent_run_id = parent_run_id
        self.model_types = model_types
        self.fit_kwargs = fit_kwargs
        self.get_model_kwargs = get_model_kwargs
        self.score_model_kwargs = score_model_kwargs
        self._best_models = self.__generate_best_model_dict()
        
    def __call__(self, trial: optuna.Trial):
        model_type = trial.suggest_categorical("model_type", self.model_types)

        estimator = get_model(trial, model_type, **self.get_model_kwargs)
        model = StaticMoveBaseModel(estimator=estimator)
        fit_kwargs = self.fit_kwargs[model_type]
        model.fit(X_train, y_train, **fit_kwargs)

        model.score(X_valid, y_valid)
        self.log_trial(trial, model)
        
        return model.balanced_accuracy
    
    def log_trial(self, trial, model: StaticMoveBaseModel):
        score = model.balanced_accuracy
        model_type = trial.params["model_type"]
        if score > self._best_models[model_type].score:
            self._best_models[model_type].model = model
            self._best_models[model_type].score = score
            self._best_models[model_type].trial = trial
        
        
        run_id = self._best_models[model_type].run_id
        with mlflow.start_run(nested=True, run_id=run_id, parent_run_id=self.parent_run_id) as model_run:
            with mlflow.start_run(nested=True, run_name=f"Trial_{trial.number}", parent_run_id=model_run.info.run_id):
                # mlflow.log_params(trial.params)
                model.log_trial()
        
    def log_best(self, trial):
        best_mode = self._best_models[trial.params["model_type"]]
        mlflow.log_params(best_mode.trial.params)
        best_mode.model.log_trial()
        
        for model_type in self.model_types:
            best_model_data = self._best_models[model_type]
            if best_model_data.model is None:
                mlflow.delete_run(best_model_data.run_id)
                continue
            with mlflow.start_run(nested=True, run_id=best_model_data.run_id, parent_run_id=self.parent_run_id):
                mlflow.log_params(best_model_data.trial.params)
                best_model_data.model.log_trial()
                
    def log_res_prediction(self, trial, x, y, set_name):
        model = self._best_models[trial.params["model_type"]].model
        log_prediction(model, x, y, set_name)
          
    def __generate_best_model_dict(self):
        best_models = {}
        for model_type in self.model_types:
            with mlflow.start_run(nested=True, run_name=model_type, parent_run_id=self.parent_run_id) as model_run:
                best_models[model_type] = BestModel(run_id=model_run.info.run_id)
        return best_models

## Study Run

In [10]:
def study_run(description, study_name, objective_kwargs):
    study = optuna.create_study(direction="maximize", study_name=study_name)
    
    with mlflow.start_run(run_name=study_name, description=description) as main_run:
        objective = Objective(main_run.info.run_id, **objective_kwargs)
        study.optimize(objective, n_trials=1, show_progress_bar=True)
        objective.log_best(study.best_trial)
        
        objective.log_res_prediction(study.best_trial, X_train, y_train, set_name="X_train")
        objective.log_res_prediction(study.best_trial, X_valid, y_valid, set_name="X_valid")
        objective.log_res_prediction(study.best_trial, X_test, y_test, set_name="X_test")
    
    return study

In [11]:
models_types = [HistGradientBoostingClassifier.__name__]

study_kwargs_1 = {
    "study_name": "NO Class Weights; NO Sample Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {HistGradientBoostingClassifier.__name__: {}},
        "get_model_kwargs": {"hist_gb_class_weight": None},
        "score_model_kwargs": {"sample_weight": None},
    }
}
study_kwargs_2 = {
    "study_name": "Class Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {HistGradientBoostingClassifier.__name__: {}},
        "get_model_kwargs": {"hist_gb_class_weight": "balanced"},
        "score_model_kwargs": {"sample_weight": None},
    }
}
study_kwargs_3 = {
    "study_name": "Sample Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {HistGradientBoostingClassifier.__name__: {"sample_weight": train_sample_weight}},
        "get_model_kwargs": {"hist_gb_class_weight": None},
        "score_model_kwargs": {"sample_weight": None},
    }
}
study_kwargs_4 = {
    "study_name": "Class Weights && Sample Weights",
    "objective_kwargs": {
        "model_types": models_types,
        "fit_kwargs": {HistGradientBoostingClassifier.__name__: {"sample_weight": train_sample_weight}},
        "get_model_kwargs": {"hist_gb_class_weight": None},
        "score_model_kwargs": {"sample_weight": valid_sample_weight},
        # "score_model_kwargs": {"sample_weight": compute_class_weight("balanced", classes=y_valid.unique(), y=y_valid)},
    }
}

In [12]:
run_description = "class/sample weights; valid with sample_weights; 10 bin; data fixed; time features added; external samples in train"

st = study_run(description=run_description, **study_kwargs_4)

[I 2024-09-05 14:07:03,861] A new study created in memory with name: Class Weights && Sample Weights


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-09-05 14:07:49,211] Trial 0 finished with value: 0.5180546132353466 and parameters: {'model_type': 'HistGradientBoostingClassifier', 'hist_learning_rate': 0.0008000211687428014, 'hist_max_iter': 917, 'hist_max_depth': 14, 'hist_min_samples_leaf': 4389, 'hist_max_leaf_nodes': 230, 'hist_max_bins': 121, 'hist_max_features': 0.834741147941946}. Best is trial 0 with value: 0.5180546132353466.


## Feature Importance

In [13]:
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance

In [14]:
best_model: HistGradientBoostingClassifier = get_model(st.best_trial, st.best_trial.params["model_type"])

In [15]:
best_model.fit(X_train, y_train)

HistGradientBoostingClassifier(interaction_cst=[{0, 9, 10}, {10, 11, 12, 13},
                                                {0, 9, 10, 11, 12, 13},
                                                {3, 4, 5, 6, 7, 8, 10},
                                                {10, 14, 15, 17, 18, 19},
                                                {16, 1, 2}],
                               learning_rate=0.0008000211687428014,
                               max_bins=121, max_depth=14,
                               max_features=0.834741147941946, max_iter=917,
                               max_leaf_nodes=230, min_samples_leaf=4389,
                               random_state=42, tol=1e-05)

In [16]:
import joblib

In [17]:
joblib.dump(best_model, "/home/tikhon/PycharmProjects/ChessWinnerPrediction/models/static_move.pkl")

['/home/tikhon/PycharmProjects/ChessWinnerPrediction/models/static_move.pkl']

In [25]:
y_pred = best_model.predict(X_test)

In [26]:
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

         0-1       0.67      0.66      0.66    294820
         1-0       0.66      0.72      0.69    302661
     1/2-1/2       0.84      0.01      0.02     28314

    accuracy                           0.66    625795
   macro avg       0.72      0.46      0.46    625795
weighted avg       0.67      0.66      0.65    625795



## show feature importance

In [27]:
result = permutation_importance(best_model, X_train, y_train, n_repeats=4, random_state=42, n_jobs=-1)

In [28]:
for i in result.importances_mean.argsort()[::-1]:
    print(f"{X_train.columns[i]}: {result.importances_mean[i]:.4f} +/- {result.importances_std[i]:.4f}")

eval: 0.1483 +/- 0.0005
EloDiff: 0.0363 +/- 0.0007
MeanElo: 0.0091 +/- 0.0001
time_diff_norm: 0.0043 +/- 0.0004
b_score: 0.0038 +/- 0.0005
white_increment_pct_in_time_per_move: 0.0034 +/- 0.0002
n_pieces: 0.0024 +/- 0.0001
is_checkmate_countdown: 0.0018 +/- 0.0004
i_move: 0.0015 +/- 0.0004
w_score: 0.0014 +/- 0.0005
BaseTime: 0.0013 +/- 0.0003
black_time_per_move: 0.0012 +/- 0.0001
white_remaining_time_norm: 0.0010 +/- 0.0001
GameDurations_norm: 0.0009 +/- 0.0001
IncrementTime: 0.0008 +/- 0.0002
black_increment_pct_in_time_per_move: 0.0007 +/- 0.0001
black_remaining_time_norm: 0.0006 +/- 0.0002
black_time_will_end_on_move: 0.0006 +/- 0.0001
white_time_per_move: 0.0005 +/- 0.0001
white_time_will_end_on_move: 0.0005 +/- 0.0002
